# Sentinel-3 OLCI workflow.
OLCI L1B → Rrs → chl-a → QC

Set paths below. Polymer needs ECMWF auxdata.

In [ ]:
import sys; sys.path.insert(0, '..')
import xarray as xr, numpy as np
from matplotlib.colors import LogNorm
from src.io import open_l1, open_geo
from src.flags import wqsf_mask
from src.plot import map_var

SCENE = '../data/S3A_OL_1_EFR____....SEN3'   # L1B
L2    = '../data/S3A_OL_2_WFR____....SEN3'   # EUMETSAT L2

## 1 · Read OLCI L1B

In [ ]:
L = open_l1(SCENE); geo = open_geo(SCENE)
L.sel(band='Oa08').plot(cmap='gray', robust=True)
L

## 2 · Polymer atmospheric correction (L1B → Rrs)

In [ ]:
from polymer.main import run_atm_corr
from polymer.level1_olci import Level1_OLCI
from polymer.level2 import Level2

run_atm_corr(Level1_OLCI(SCENE), Level2(filename='l2_polymer.nc', fmt='netcdf4'))

## 3 · OC4Me chl-a (Morel et al. 2007)
max(Rrs443,490,510)/Rrs560 blue-green ratio.

In [ ]:
ds = xr.open_dataset('l2_polymer.nc')
print(list(ds.data_vars))  # confirm Rrs names

R = np.log10(xr.concat([ds.Rrs443, ds.Rrs490, ds.Rrs510], dim='b').max('b') / ds.Rrs560)
chl = 10 ** (0.4502 - 2.9337*R + 1.6876*R**2 - 0.4321*R**3)
chl.attrs['units'] = 'mg m-3'
chl.plot(norm=LogNorm(0.03, 30), cmap='viridis')

## 4 · EUMETSAT L2 WQSF masking

In [ ]:
chl_l2 = xr.open_dataset(f'{L2}/chl_oc4me.nc').chl_oc4me
wq = xr.open_dataset(f'{L2}/wqsf.nc').WQSF
geo2 = xr.open_dataset(f'{L2}/geo_coordinates.nc')

bad = wqsf_mask(wq)
print(f'masked: {bad.mean().values:.1%}')
map_var(chl_l2.where(~bad), geo2.latitude, geo2.longitude,
        vmin=0.03, vmax=30, title='chl_oc4me (WQSF-masked)')

 RGB quicklook (L1B)

In [ ]:
import matplotlib.pyplot as plt

rgb = np.dstack([L.sel(band=b).values for b in ('Oa08', 'Oa06', 'Oa03')])  # 665/560/442
rgb = np.clip(rgb / np.nanpercentile(rgb, 98, axis=(0, 1)), 0, 1) ** 0.5   # stretch + gamma
plt.figure(figsize=(6, 8))
plt.imshow(rgb)
plt.axis('off'); plt.title('OLCI RGB')

Rrs spectrum (centre pixel, Polymer output)

In [ ]:
px = ds.isel({d: s // 2 for d, s in ds.sizes.items()})
rrs = {v: float(px[v]) for v in ds.data_vars if str(v).startswith('Rrs')}
wl = sorted(int(v[3:]) for v in rrs)

plt.plot(wl, [rrs[f'Rrs{w}'] for w in wl], 'o-')
plt.xlabel('Wavelength (nm)'); plt.ylabel('Rrs (sr$^{-1}$)')
plt.title('Rrs spectrum'); plt.grid(alpha=0.3)

Chl-*a* map (log scale, geolocated)

In [ ]:
map_var(chl.where(chl > 0), geo.latitude, geo.longitude,
        norm=LogNorm(0.03, 30), cmap='viridis', title='OC4Me chl-a (mg m$^{-3}$)')

L1B Lt map

In [ ]:
plt.figure()
Lt = L.sel(band='Oa11')
map_var(Lt, geo.latitude, geo.longitude, cmap='magma',
        vmin=float(np.nanpercentile(Lt, 2)), vmax=float(np.nanpercentile(Lt, 98)),
        title='OLCI Lt Oa11 (708.75 nm)')

Polymer Rrs + ratio maps

In [ ]:
lat = ds.get('latitude', ds.get('lat', geo.latitude))
lon = ds.get('longitude', ds.get('lon', geo.longitude))

plt.figure(); map_var(ds['Rrs560'], lat, lon, cmap='turbo', vmin=0,
                      vmax=float(np.nanpercentile(ds['Rrs560'], 98)), title='Polymer Rrs560')
plt.figure(); map_var(R, lat, lon, cmap='coolwarm', vmin=-1, vmax=1,
                      title='OC4Me log10 blue-green ratio')

Chl-*a* histogram

In [ ]:
chl.where(chl > 0).plot.hist(bins=80, range=(0, 5))
plt.xlabel('chl-a (mg m$^{-3}$)'); plt.title('OC4Me chl-a distribution')

WQSF QC pie chart

In [ ]:
plt.figure()
plt.pie([int((~bad).sum()), int(bad.sum())], labels=['valid', 'masked'],
        autopct='%1.1f%%', colors=['#2ca02c', '#d62728'])
plt.title('WQSF QC')